# PyroPredict — Phase 4: Ablation Experiments

**CMPE 258 Deep Learning** | Spring 2026 | SJSU

This notebook runs **4 targeted ablation experiments** on top of the YOLO26m
baseline (the stronger of the two baselines) to demonstrate meaningful
improvement via training-strategy changes.

### Baseline to beat (from Phase 3)

| Model | mAP@50 | mAP@50:95 | Precision | Recall | F1 |
|-------|--------|-----------|-----------|--------|-----|
| YOLO26m baseline | 0.7760 | 0.4461 | 0.7726 | 0.7020 | 0.7356 |

### Experiments

| # | Experiment | What it targets |
|---|-----------|----------------|
| 1 | Domain augmentations (smoke/fire-specific) | Low recall — more variation helps model detect more |
| 2 | Hard-negative tuning (reduce negative ratio) | 45.7% negatives may overwhelm the model |
| 3 | Multi-scale training + longer schedule | Low mAP@50:95 — better box localization |
| 4 | Combined best settings | Stack all improvements |

**Runtime:** GPU required (L4 recommended)  
**Time:** ~3–4 hours total (4 training runs)

## 0 — Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless seaborn pyyaml kaggle onnx onnxruntime albumentations

import torch, os, json, shutil, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime → Change runtime type → GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/PyroPredict'
DRIVE_RUNS    = f'{DRIVE_PROJECT}/runs'
DRIVE_EXPORTS = f'{DRIVE_PROJECT}/exports'
DRIVE_METRICS = f'{DRIVE_PROJECT}/metrics'
for d in [DRIVE_RUNS, DRIVE_EXPORTS, DRIVE_METRICS]:
    os.makedirs(d, exist_ok=True)

## 1 — Download dataset

In [ ]:
# ┌──────────────────────────────────────────────────────┐
# │  PASTE YOUR KAGGLE USERNAME AND TOKEN BELOW          │
# └──────────────────────────────────────────────────────┘
KAGGLE_USERNAME = "your_username_here"
KAGGLE_TOKEN    = "your_token_here"

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

In [ ]:
%%time
DATA_DIR = '/content/dfire'
if not os.path.exists(DATA_DIR):
    !kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content/
    !unzip -q /content/smoke-fire-detection-yolo.zip -d {DATA_DIR}
    !rm -f /content/smoke-fire-detection-yolo.zip
    print(f'Downloaded to {DATA_DIR}')
else:
    print(f'Already exists at {DATA_DIR}')

In [ ]:
import yaml

data_root = Path(DATA_DIR)

def find_split_dirs(root):
    splits = {}
    for split_name in ['train', 'valid', 'val', 'test']:
        for candidate in [root / split_name / 'images', root / 'images' / split_name]:
            if candidate.is_dir() and any(candidate.iterdir()):
                canon = 'val' if split_name == 'valid' else split_name
                splits[canon] = candidate
                break
    if not splits:
        for child in root.iterdir():
            if child.is_dir():
                deeper = find_split_dirs(child)
                if deeper:
                    return deeper
    return splits

split_img_dirs = find_split_dirs(data_root)

dataset_yaml = data_root / 'dataset.yaml'
yaml_cfg = {
    'path': str(data_root.resolve()),
    'train': str(split_img_dirs['train'].relative_to(data_root)),
    'val':   str(split_img_dirs['val'].relative_to(data_root)),
    'test':  str(split_img_dirs['test'].relative_to(data_root)) if 'test' in split_img_dirs else '',
    'nc': 2,
    'names': ['fire', 'smoke'],
}
with open(dataset_yaml, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False, sort_keys=False)

print(f'dataset.yaml ready at {dataset_yaml}')
print(f'Train images: {len(list(split_img_dirs["train"].glob("*")))}')

## 2 — Baseline reference numbers

In [ ]:
# Phase 3 baseline results (hardcoded for comparison)
BASELINE = {
    'Model': 'YOLO26m (baseline)',
    'mAP@50': 0.7760,
    'mAP@50:95': 0.4461,
    'Precision': 0.7726,
    'Recall': 0.7020,
    'F1': 0.7356,
}

print('Baseline to beat:')
for k, v in BASELINE.items():
    print(f'  {k}: {v}')

## 3 — Shared training config

Base config shared across all ablations. Each experiment overrides specific fields.

In [ ]:
from ultralytics import YOLO

BASE_CFG = dict(
    data      = str(dataset_yaml),
    epochs    = 20,         # reduced from 30 to fit in Colab session window
    imgsz     = 640,
    batch     = 16,         # reduced from 32 to avoid OOM
    patience  = 8,
    optimizer = 'SGD',
    lr0       = 0.01,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    warmup_momentum = 0.8,
    cos_lr    = True,
    workers   = 2,          # reduced for stability
    seed      = 42,
    val       = True,
    plots     = True,
    verbose   = True,
)

def run_experiment(name, overrides):
    """Train YOLO26m with overrides. Resilient to disconnects:
       - If results already exist on Drive, skip and load them
       - After training, immediately copy weights + metrics to Drive
    """
    drive_run_dir = Path(f'{DRIVE_RUNS}/{name}')
    drive_result_file = Path(f'{DRIVE_METRICS}/{name}_result.json')

    # ── RESUME: skip if already complete ──
    if drive_result_file.exists():
        print(f'\n[SKIP] {name} already complete — loading saved results')
        with open(drive_result_file) as f:
            return json.load(f), None

    cfg = {**BASE_CFG, **overrides}
    print(f'\n{"═"*60}')
    print(f'EXPERIMENT: {name}')
    print(f'{"═"*60}')
    for k, v in overrides.items():
        print(f'  override  {k}: {v}')

    model = YOLO('yolo26m.pt')
    t0 = time.time()
    model.train(**cfg, project='/content/runs', name=name)
    train_time = time.time() - t0

    best = YOLO(f'/content/runs/{name}/weights/best.pt')
    metrics = best.val(data=str(dataset_yaml), split='test')

    mp = metrics.box.mp
    mr = metrics.box.mr
    result = {
        'Model': name,
        'mAP@50': float(metrics.box.map50),
        'mAP@50:95': float(metrics.box.map),
        'Precision': float(mp),
        'Recall': float(mr),
        'F1': float(2 * mp * mr / (mp + mr + 1e-8)),
        'Train time (min)': train_time / 60,
    }

    # ── PERSIST IMMEDIATELY: copy run dir + result JSON to Drive ──
    src = Path(f'/content/runs/{name}')
    if src.exists():
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(src, drive_run_dir)
        print(f'  → Run saved to {drive_run_dir}')

    with open(drive_result_file, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  → Metrics saved to {drive_result_file}')

    print(f'\n  mAP@50: {result["mAP@50"]:.4f}  (baseline: {BASELINE["mAP@50"]:.4f}  Δ {result["mAP@50"] - BASELINE["mAP@50"]:+.4f})')
    print(f'  Recall: {result["Recall"]:.4f}  (baseline: {BASELINE["Recall"]:.4f}  Δ {result["Recall"] - BASELINE["Recall"]:+.4f})')
    print(f'  Time:   {result["Train time (min)"]:.1f} min')
    return result, best

all_results = [BASELINE.copy()]
all_results[0]['Train time (min)'] = '-'

## 4 — Experiment 1: Domain-specific augmentations

**Target:** Low recall (0.702). The model misses detections because it hasn't
seen enough variation. We boost augmentation aggressiveness:
- Stronger mosaic + mixup
- HSV shifts tuned for smoke (gray/white) and fire (orange/red)
- Copy-paste augmentation to place smoke in clean backgrounds
- Scale variation for small/distant smoke plumes

In [ ]:
result_1, _ = run_experiment('ablation_1_augment', {
    'mosaic':    1.0,      # probability of mosaic (default 1.0, ensure it's on)
    'mixup':     0.15,     # baseline default is 0.0 — adds blended image pairs
    'copy_paste': 0.15,    # paste objects from other images onto backgrounds
    'hsv_h':     0.02,     # hue shift — slightly higher for fire/smoke variation
    'hsv_s':     0.75,     # saturation — wider range for smoky/hazy conditions
    'hsv_v':     0.50,     # value/brightness — dawn/dusk/night variation
    'scale':     0.7,      # zoom in/out — helps with distant smoke plumes
    'flipud':    0.1,      # vertical flip — mild, smoke rises upward
    'fliplr':    0.5,      # horizontal flip
    'erasing':   0.1,      # random erasing — regularisation
})
all_results.append(result_1)

## 5 — Experiment 2: Hard-negative tuning

**Target:** 45.7% of training images are negatives (no fire/smoke). This can
overwhelm the model. We reduce the background class weight and add
label smoothing to soften the learning signal.

In [ ]:
result_2, _ = run_experiment('ablation_2_negatives', {
    'label_smoothing': 0.05,   # soften hard labels → reduces overconfidence
    'cls':             1.0,    # classification loss weight (bump slightly)
    'box':             8.0,    # box loss weight (bump to prioritise localization)
})
all_results.append(result_2)

## 6 — Experiment 3: Multi-scale + longer training

**Target:** Low mAP@50:95 (0.446). This means the predicted boxes don't
tightly align with ground truth at strict IoU thresholds. Multi-scale
training and a longer schedule with lower learning rate help.

In [ ]:
result_3, _ = run_experiment('ablation_3_multiscale', {
    'scale':     0.9,      # aggressive multi-scale (±90% zoom)
    'lr0':       0.008,    # slightly lower LR for fine-grained learning
    'epochs':    25,       # slightly longer schedule
    'patience':  10,
})
all_results.append(result_3)

## 7 — Experiment 4: Combined best settings

Stack the improvements from all three experiments.

In [ ]:
result_4, best_model = run_experiment('ablation_4_combined', {
    # From Exp 1: augmentations
    'mixup':       0.15,
    'copy_paste':  0.15,
    'hsv_h':       0.02,
    'hsv_s':       0.75,
    'hsv_v':       0.50,
    'scale':       0.9,
    'flipud':      0.1,
    'erasing':     0.1,
    # From Exp 2: loss tuning
    'label_smoothing': 0.05,
    'box':         8.0,
    # From Exp 3: schedule
    'lr0':         0.008,
    'epochs':      25,
    'patience':    10,
})
all_results.append(result_4)

## 8 — Ablation summary table

In [ ]:
df_ablation = pd.DataFrame(all_results)

# Add delta columns
for col in ['mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']:
    df_ablation[f'Δ{col}'] = df_ablation[col].apply(
        lambda x: x - BASELINE[col] if isinstance(x, (int, float)) else 0
    )

print('\n' + '═'*80)
print('ABLATION RESULTS')
print('═'*80)
display_cols = ['Model', 'mAP@50', 'ΔmAP@50', 'mAP@50:95', 'ΔmAP@50:95',
                'Precision', 'Recall', 'F1', 'Train time (min)']
print(df_ablation[display_cols].to_string(
    index=False,
    float_format='{:.4f}'.format,
))
print('═'*80)

In [ ]:
# Ablation bar chart
metric_cols = ['mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']
models = df_ablation['Model'].tolist()
colors = ['#94a3b8', '#3b82f6', '#f59e0b', '#10b981', '#ef4444']

x = np.arange(len(metric_cols))
width = 0.15
offsets = np.linspace(-(len(models)-1)*width/2, (len(models)-1)*width/2, len(models))

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model, color) in enumerate(zip(models, colors)):
    row = df_ablation[df_ablation['Model'] == model].iloc[0]
    vals = [row[c] for c in metric_cols]
    bars = ax.bar(x + offsets[i], vals, width, label=model, color=color)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
                ha='center', va='bottom', fontsize=6, rotation=45)

ax.set_ylabel('Score')
ax.set_title('Ablation Study — Incremental Improvements over Baseline')
ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=8)
fig.tight_layout()
fig.savefig(f'{DRIVE_METRICS}/ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 — Export best ablation model to ONNX

In [ ]:
# Export the combined model (ablation 4) — our best model
# If resumed (best_model is None), reload best.pt from Drive
if best_model is None:
    drive_best = Path(f'{DRIVE_RUNS}/ablation_4_combined/weights/best.pt')
    if drive_best.exists():
        print(f'Loading best.pt from Drive: {drive_best}')
        best_model = YOLO(str(drive_best))
    else:
        raise FileNotFoundError('No best.pt found. Re-run experiment 4.')

print('Exporting best ablation model to ONNX...')
best_onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
print(f'  → {best_onnx_path}')
size_mb = os.path.getsize(best_onnx_path) / (1024 * 1024)
print(f'  Size: {size_mb:.1f} MB')

## 10 — Save everything to Google Drive

In [ ]:
# Save all ablation runs
for name in ['ablation_1_augment', 'ablation_2_negatives',
             'ablation_3_multiscale', 'ablation_4_combined']:
    src = Path(f'/content/runs/{name}')
    dst = Path(f'{DRIVE_RUNS}/{name}')
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Saved {name} → {dst}')

# Save best ONNX
dst_onnx = f'{DRIVE_EXPORTS}/ablation_4_combined.onnx'
shutil.copy2(best_onnx_path, dst_onnx)
print(f'Saved ONNX → {dst_onnx}')

# Save metrics
df_ablation.to_csv(f'{DRIVE_METRICS}/ablation_results.csv', index=False)
print(f'Saved metrics → {DRIVE_METRICS}/ablation_results.csv')

In [ ]:
# Final summary
print(f'\n{"═"*80}')
print('PHASE 4 COMPLETE — ABLATION EXPERIMENTS')
print(f'{"═"*80}')

print('\nFull ablation table:')
display_cols = ['Model', 'mAP@50', 'ΔmAP@50', 'mAP@50:95', 'ΔmAP@50:95',
                'Recall', 'F1', 'Train time (min)']
print(df_ablation[display_cols].to_string(
    index=False, float_format='{:.4f}'.format,
))

# Highlight best improvement
best_row = df_ablation.loc[df_ablation['mAP@50'].idxmax()]
delta = best_row['mAP@50'] - BASELINE['mAP@50']
print(f'\nBest model: {best_row["Model"]}')
print(f'  mAP@50 improvement: {BASELINE["mAP@50"]:.4f} → {best_row["mAP@50"]:.4f} ({delta:+.4f})')
print(f'\nAll artifacts saved to: {DRIVE_PROJECT}')
print(f'{"═"*80}')
print('\nNext → Phase 5: ONNX INT8 quantization & efficiency benchmarking')